# Merge Augmented Data
Combina il dataset originale con i campioni sintetici generati (STYLE_GAN_ADA o WGAN-GP), controlla la distribuzione e salva un nuovo `dataset_augmented.h5` pronto per il training.

## Section 1: Monta Google Drive
Necessario per leggere l'HDF5 originale e i sintetici su Drive.

In [ ]:
USE_COLAB = False

if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('Google Drive mounted')
else:
    print('Running outside Colab: skip Drive mount')

## Section 2: Config path sintetici (DISGUST + SURPRISE)
Imposta i percorsi e prepara l'elenco delle cartelle sintetiche da unire (`STYLE_GAN_augmentation_disgust` e `STYLE_GAN_augmentation_surprise`).

In [ ]:
import numpy as np
import h5py, os
import matplotlib.pyplot as plt

USE_COLAB = False

if USE_COLAB:
    ORIG_PATH = '/content/drive/MyDrive/final_scripts/dataset/dataset.h5'
    SYN_DIR_DISGUST = '/content/drive/MyDrive/STYLE_GAN_augmentation_disgust'
    SYN_DIR_SURPRISE = '/content/drive/MyDrive/STYLE_GAN_augmentation_surprise'
    OUT_PATH = '/content/drive/MyDrive/final_scripts/dataset/dataset_augmented.h5'
else:
    ORIG_PATH = os.path.expanduser('~/data/dataset.h5')
    SYN_DIR_DISGUST = os.path.expanduser('~/models/STYLE_GAN_augmentation_disgust')
    SYN_DIR_SURPRISE = os.path.expanduser('~/models/STYLE_GAN_augmentation_surprise')
    OUT_PATH = os.path.expanduser('~/data/dataset_augmented.h5')

SYN_DIRS = [SYN_DIR_DISGUST, SYN_DIR_SURPRISE]

print('Using original dataset:', ORIG_PATH)
print('Using synthetic dirs:')
for d in SYN_DIRS:
    print(' -', d)
print('Output augmented dataset:', OUT_PATH)

## Section 3: Carica dataset originale
Legge train/val dall'HDF5 di origine per avere riferimento e class names.

In [ ]:
# Load original
with h5py.File(ORIG_PATH, 'r') as f:
    X_train = np.array(f['X_train'])
    y_train = np.array(f['y_train'])
    X_val = np.array(f['X_val'])
    y_val = np.array(f['y_val'])
    class_names = [c.decode('utf-8') for c in f['class_names']]
print('Original shapes', X_train.shape, y_train.shape, X_val.shape, y_val.shape)

## Section 4: Carica sintetici (DISGUST + SURPRISE) e rimappa etichette
Carica i `.npy` da entrambe le cartelle sintetiche, rimappa le etichette locali agli indici globali e concatena tutto in un unico blocco.

In [ ]:
# Load synthetic from multiple dirs (DISGUST + SURPRISE)
all_synth_images = []
all_synth_labels_global = []

for syn_dir in SYN_DIRS:
    synth_images = np.load(os.path.join(syn_dir, 'synthetic_images.npy'))
    synth_labels = np.load(os.path.join(syn_dir, 'synthetic_labels.npy'))
    rare_idx = np.load(os.path.join(syn_dir, 'rare_class_indices.npy'))

    print(f'[{os.path.basename(syn_dir)}] Synthetic', synth_images.shape, synth_labels.shape, 'rare_idx=', rare_idx)

    # Map synthetic labels (local) back to global class ids
    label_map_rev = {local: int(global_id) for local, global_id in enumerate(rare_idx)}
    synth_labels_global = np.array([label_map_rev[int(v)] for v in synth_labels], dtype=np.int32)

    # Normalize back to [0,1] float32 for training
    synth_images_float = synth_images.astype('float32') / 255.0

    all_synth_images.append(synth_images_float)
    all_synth_labels_global.append(synth_labels_global)

synth_images_float = np.concatenate(all_synth_images, axis=0)
synth_labels_global = np.concatenate(all_synth_labels_global, axis=0)

print('Merged synthetic shapes:', synth_images_float.shape, synth_labels_global.shape)

## Section 5: Merge e shuffle
Concatena train originale + sintetici, poi mescola per evitare ordini di blocco.

In [ ]:
# Concatenate with original train split
X_train_new = np.concatenate([X_train.astype('float32')/255.0, synth_images_float])
y_train_new = np.concatenate([y_train, synth_labels_global])
# Keep val/test unchanged
print('New train shape', X_train_new.shape, y_train_new.shape)

# Shuffle
idx = np.random.permutation(len(X_train_new))
X_train_new = X_train_new[idx]
y_train_new = y_train_new[idx]

## Section 6: Controllo distribuzione
Verifica i conteggi di classe dopo il merge per valutare il riequilibrio.

In [ ]:
# Quick distribution check
import collections
cnt = collections.Counter(y_train_new)
print('Class counts after merge:')
for i, name in enumerate(class_names):
    print(f'{name:10s}: {cnt[i]}')
plt.bar(class_names, [cnt[i] for i in range(len(class_names))])
plt.xticks(rotation=45)
plt.show()

## Section 7: Salvataggio dataset fuso
Salva `dataset_augmented.h5` con train (originale+synthetic) e val invariato.

In [ ]:
# Save new dataset
with h5py.File(OUT_PATH, 'w') as f:
    f.create_dataset('X_train', data=X_train_new, compression='gzip')
    f.create_dataset('y_train', data=y_train_new, compression='gzip')
    f.create_dataset('X_val', data=X_val.astype('float32')/255.0, compression='gzip')
    f.create_dataset('y_val', data=y_val, compression='gzip')
    f.create_dataset('class_names', data=np.array(class_names, dtype='S'))
print('Saved', OUT_PATH)